In [22]:
from pyscf import gto, scf, cc

a = 2.2 # 2aB
nH = 2
atoms = ""
for i in range(nH):
    atoms += f"N {i*a:.5f} 0.00000 0.00000 \n"

mol = gto.M(atom=atoms, basis="sto6g", unit='B', spin=0, verbose=4)
mol.build()

mf = scf.RHF(mol)
mf.kernel()

mo = mf.stability()[0]
dm = mf.make_rdm1(mo,mf.mo_occ)
mf.kernel(dm0=dm)
mo = mf.stability()[0]
dm = mf.make_rdm1(mo,mf.mo_occ)
mf.kernel(dm0=dm)

nfrozen = 0
mycc = cc.CCSD(mf,frozen=nfrozen)
mycc.kernel()
# et = mycc.ccsd_t()
# print(mycc.e_tot + et)
mycc.energy()

System: uname_result(system='Linux', node='sharmagroup-rn', release='6.17.0-14-generic', version='#14~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Thu Jan 15 15:52:10 UTC 2', machine='x86_64')  Threads 16
Python 3.11.14 (main, Oct 21 2025, 18:31:21) [GCC 11.2.0]
numpy 2.3.1  scipy 1.16.2  h5py 3.14.0
Date: Tue Feb 17 17:46:53 2026
PySCF version 2.11.0
PySCF path  /home/sharmagroup/sharmagroup/pyscf
GIT HEAD (branch master) 3d1768f5e33b144b606c3d2c81c12ee54d794501

[ENV] PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 2
[INPUT] num. electrons = 14
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = B
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 N      0.000000000000   0.000000000000   0.000000000000 AA    0.000000000000   0.000000000000   0.000000000000 Bohr 

np.float64(-0.1742896316098625)

In [42]:
# example for PT2
options = {'n_eql': 3,
           'n_prop_steps': 50,
            'n_ene_blocks': 1,
            'n_sr_blocks': 5,
            'n_blocks': 50,
            'n_walkers': 100,
            'seed': 2,
            'walker_type': 'rhf',
            'trial': 'ccsd_pt2',
            'dt':0.005,
            'free_projection':False,
            'fp_abs': False,
            'group': False,
            'ad_mode': None,
            'use_gpu': False,
            }

from ad_afqmc.prop_unrestricted import prep
import jax
jax.config.update("jax_enable_x64", True)
prep.prep_afqmc(mycc,chol_cut=1e-5)
# prop_unrestricted.run_afqmc(options,nproc=1)
option_file='options.bin'
import pickle
with open(option_file, 'wb') as f:
    pickle.dump(options, f)

#
# Preparing AFQMC calculation
# Calculating Cholesky integrals
# Finished calculating Cholesky integrals
#
# Size of the correlation space:
# Number of electrons: (7, 7)
# Number of basis functions: 10
# Number of Cholesky vectors: 42
#


In [43]:
import time
import numpy as np
from jax import random
from jax import numpy as jnp
from functools import partial 

ham_data, ham, prop, trial, wave_data, sampler, options = (prep._prep_afqmc())

init_time = time.time()

### initialize propagation
init_walkers = None
trial_rdm1 = trial.get_rdm1(wave_data)
if "rdm1" not in wave_data:
    wave_data["rdm1"] = trial_rdm1
ham_data = ham.build_measurement_intermediates(ham_data, trial, wave_data)
ham_data = ham.build_propagation_intermediates(ham_data, prop, trial, wave_data)

prop_data = prop.init_prop_data(trial, wave_data, ham_data, init_walkers)
if jnp.abs(jnp.sum(prop_data["overlaps"])) < 1.0e-6:
    raise ValueError(
        "Initial overlaps are zero. Pass walkers with non-zero overlap."
    )
prop_data["key"] = random.PRNGKey(options["seed"])

prop_data["overlaps"] = trial.calc_overlap(prop_data["walkers"], wave_data)
prop_data["n_killed_walkers"] = 0

e_init= jnp.real(trial.calc_energy(prop_data["walkers"], ham_data, wave_data)[0])
# ept_sp = h0 + e0/t1 + e1/t1 - t2 * e0 / t1**2
# ept = jnp.array(jnp.sum(ept_sp) / prop.n_walkers)
prop_data["e_estimate"] = e_init
# eci = trial.calc_energy(
#     prop_data['walkers'], ham_data, wave_data)
prop_data["pop_control_ene_shift"] = prop_data["e_estimate"]

print(e_init)
print(e_init-mf.e_tot)

# norb: 10
# nelec: (7, 7)
# n_eql: 3
# n_prop_steps: 50
# n_ene_blocks: 1
# n_sr_blocks: 5
# n_blocks: 50
# n_walkers: 100
# seed: 2
# walker_type: rhf
# trial: ccsd_pt2
# dt: 0.005
# free_projection: False
# fp_abs: False
# group: False
# use_gpu: False
# n_exp_terms: 6
# symmetry: False
# save_walkers: False
# n_batch: 1
# max_error: 0.001
-108.5448636405666
9.501808847289794e-06


In [25]:
from ad_afqmc.prop_unrestricted import wavefunctions_restricted
amp_file="amplitudes.npz"
amplitudes = np.load(amp_file)
t1 = jnp.array(amplitudes["t1"])
t2 = jnp.array(amplitudes["t2"])
wave_data["t1"] = t1
wave_data["t2"] = t2

trial_stocc = wavefunctions_restricted.stoccsd(trial.norb, trial.nelec, n_batch=options["n_batch"])
trial_stocc.nslater = 100
wave_data['stocc'] = trial_stocc.get_stocc(wave_data, prop_data)
e_stocc = trial.calc_energy(wave_data['stocc'], ham_data, wave_data)
o_stocc = trial.calc_overlap(wave_data['stocc'], wave_data)
e_stocc = jnp.sum(o_stocc * e_stocc) / jnp.sum(o_stocc)

print(e_stocc-mycc.e_tot)

(0.0055306449235814625-0.0002594484032707563j)


In [100]:
# energy evaluator for <exp(T1)HF|(xtau + 1/2(xtau)^2)H|walker>
# exploitate the speed up by disconnected doubles

from jax import jit, lax, jvp
import opt_einsum as oe

# ad reference 
@jit
def _green(walker: jax.Array, wave_data: dict) -> jax.Array:
    '''
    full green's function 
    <exp(T1)HF|a_p^dagger a_q|walker>/<exp(T1)HF|walker>
    '''
    slater = wave_data["mo_t"]
    green = (walker @ (
        jnp.linalg.inv(slater.T.conj() @ walker)
        ) @ slater.T.conj()).T
    return green

@jit
def _walker_olp(walker, wave_data):
    ''' 
    <exp(T1)HF|walker>
    '''
    olp = jnp.linalg.det(wave_data["mo_t"].T.conj() @ walker) ** 2
    return olp

@jit
def _ci_walker_olp(walker, wave_data, ci1, ci2) -> complex:
    ''' 
    <exp(T1)HF|1+ci1+ci2|walker>
    = c_ia <exp(T1)HF|ia|walker> + 1/2 c_iajb <exp(T1)HF|ijab|walker>
    '''
    nocc = walker.shape[1]
    green_ov = _green(walker, wave_data)[:nocc, nocc:]
    o0 = _walker_olp(walker, wave_data)
    o1 = 2 * oe.contract("ia,ia-> ", ci1, green_ov, backend="jax")
    o2 = 2 * oe.contract("iajb,ia,jb->", ci2, green_ov, green_ov, backend="jax") \
        - oe.contract("iajb,ib,ja->", ci2, green_ov, green_ov, backend="jax")
    return (1.0 + o1 + o2) * o0

@jit
def _ci_exp_h1(x, h1_mod, walker, wave_data, ci1, ci2) -> complex:
    '''
    <exp(T1)HF|(1+ci1+ci2) exp(x*h1_mod)|walker>
    '''
    t = x * h1_mod
    walker_1x = walker + t.dot(walker)
    o_exp = _ci_walker_olp(walker_1x, wave_data, ci1, ci2)
    # o0 =_calc_overlap_restricted(walker, wave_data)
    return o_exp 

@jit
def _ci_exp_h2(x, chol_i, walker, wave_data, ci1, ci2) -> complex:
    '''
    <exp(T1)HF|(1+ci1+ci2) exp(x*h2)|walker>
    '''
    walker_2x = (
            walker
            + x * chol_i.dot(walker)
            + x**2 / 2.0 * chol_i.dot(chol_i.dot(walker))
        )
    o_exp = _ci_walker_olp(walker_2x, wave_data, ci1, ci2)
    # o0 = self._calc_overlap_restricted(walker, wave_data)
    return o_exp

@jit
def _d2_ci_exp_h2i(chol_i, walker, wave_data, ci1, ci2):
    x = 0.0
    f = lambda a: _ci_exp_h2(a, chol_i, walker, wave_data, ci1, ci2)
    _, d2f = jax.jvp(lambda x: jax.jvp(f, [x], [1.0])[1], [x], [1.0])
    return d2f


@partial(jit, static_argnums=0)
def _calc_energy_ad(trial, walker, ham_data, wave_data, ci1, ci2):

    norb = trial.norb
    h0 = ham_data['h0']
    h1_mod = ham_data['h1_mod']
    chol = ham_data["chol"].reshape(-1, norb, norb)

    o0 = _walker_olp(walker, wave_data)

    # one body
    f1 = lambda a: _ci_exp_h1(a, h1_mod, walker, wave_data, ci1, ci2)
    olp, d1_overlap = jvp(f1, [0.0], [1.0])

    # two body
    def scan_chol(carry, c):
        walker, wave_data = carry
        return carry, _d2_ci_exp_h2i(c, walker, wave_data, ci1, ci2)

    _, d2_overlap_i = lax.scan(scan_chol, (walker, wave_data), chol)
    d2_overlap = jnp.sum(d2_overlap_i)/2

    # <psi|(1+ci1+ci2) (h1+h2)|walker> / <psi|1+ci1+ci2|walker>
    e12 = (d1_overlap + d2_overlap) / olp

    return olp/o0, h0 + e12

In [44]:
# walker = np.random.rand(*prop_data["walkers"][0].shape)
# walker = jnp.array(walker)
walker = jnp.eye(trial.norb)[:,:trial.nelec[0]]
# wave_data['mo_t'] = jnp.eye(trial.norb)[:,:trial.nelec[0]]
print(walker)
t1 = wave_data['t1'] * 0
t2 = wave_data['t2'] #+ jnp.einsum("ia,jb->iajb", t1, t1)
energy_ad = _calc_energy_ad(trial, walker, ham_data, wave_data, t1, t2)
print(energy_ad-mycc.e_tot)

[[1. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]]
9.64489760235665e-06


In [ ]:
@partial(jit, static_argnums=0)
def _calc_energy_tc(
    trial, 
    walker: jax.Array, 
    ham_data: dict, 
    wave_data: dict,
    ci1:  jax.Array,
    ci2:  jax.Array,
    ):

    '''
    A local energy evaluator for <psi(ci1+ci2)HF|H|walker> / <psi(ci1+ci2)|walker>
    all operators and the walkers and psi are in the same basis (normally MO)
    |psi> is not necesarily diagonal
    
    all green's function and the chol and ci coeff are as their original definition
    no half rotation performed
    '''

    nocc = trial.nelec[0]
    h0  = ham_data['h0']
    # mo_t = wave_data["mo_t"]
    chol = ham_data["chol"].reshape(-1, trial.norb, trial.norb)
    green = _green(walker, wave_data) #(walker @ (jnp.linalg.inv(mo_t.T @ walker)) @ mo_t.T).T
    green_ov = green[:nocc, nocc:]
    greenp = (green - jnp.eye(trial.norb))[:, nocc:]

    h1 = (ham_data["h1"][0] + ham_data["h1"][1]) / 2.0
    
    ##################### ref terms #########################
    # one-body 
    hg = oe.contract("pq,pq->", h1, green, backend="jax")
    e1_0 = 2 * hg

    # two-body 
    gl = oe.contract("pr,gqr->gpq", green, chol, backend="jax")
    trgl = oe.contract('gpp->g', gl, backend="jax")
    e2_0_1 = 2 * jnp.sum(trgl**2)
    e2_0_2 = -oe.contract('gpq,gqp->',gl,gl, backend="jax")
    e2_0 = e2_0_1 + e2_0_2
    ##########################################################

    ######################### ci terms #######################
    # one-body single excitations 
    ci1g = oe.contract("ia,ia->", ci1, green_ov, backend="jax")
    e1_1_1 = 4 * ci1g * hg # c_ia G_ia G_pq h_pq
    gpci1 = oe.contract("pa,ia->pi", greenp, ci1, backend="jax")
    ci1_green = oe.contract("pi,iq->pq", gpci1, green[:nocc,:], backend="jax")
    e1_1_2 = -2 * oe.contract("pq,pq->", h1, ci1_green, backend="jax") # c_ia Gp_pa G_iq h_pq
    e1_1 = e1_1_1 + e1_1_2 # <psi|ci1 h1|walker> / <psi|walker>

    # one-body double excitations
    t2g_c = oe.contract("iajb,ia->jb", ci2, green_ov, backend="jax")
    t2g_e = oe.contract("iajb,ib->ja", ci2, green_ov, backend="jax")
    t2_green_c = (greenp @ t2g_c.T) @ green[:nocc,:]
    t2_green_e = (greenp @ t2g_e.T) @ green[:nocc,:]
    t2_green = 2 * t2_green_c - t2_green_e
    t2g = 2 * t2g_c - t2g_e
    gt2g = oe.contract("ia,ia->", t2g, green_ov, backend="jax")
    e1_2_1 = 2 * hg * gt2g
    e1_2_2 = -2 * oe.contract("ij,ij->", h1, t2_green, backend="jax")
    e1_2 = e1_2_1 + e1_2_2 # <exp(T1)HF|T2 h1|walker> / <exp(T1)HF|walker>

    # two-body single excitations
    e2_1_1 = 2 * e2_0 * ci1g # c_ia G_ia G_pr G_ps L_pr L_ps
    lci1g = oe.contract("gpq,pq->g", chol, ci1_green, backend="jax")
    e2_1_2 = -2 * oe.contract("g,g->",lci1g, trgl, backend="jax") # c_ia Gp_pa G_ir G_qs L_pr L_qs

    lci1 = oe.contract("gpa,ia->gpi", chol[:, :, nocc:], ci1, backend="jax")
    lg1 = oe.contract("gpr,qr->gpq", chol, green[:nocc,:], backend="jax")
    lci1g = oe.contract("gri,pr->gip", lci1, green, backend="jax")
    glgpci1 = jnp.einsum("gpr,ri->gpi", gl, gpci1, optimize="optimal")
    e2_1_3 = jnp.einsum("gpi,gpi->", glgpci1, lg1, optimize="optimal")
    e2_1 = e2_1_1 + 2 * (e2_1_2 + e2_1_3) # <exp(T1)HF|ci1 h2|walker> / <exp(T1)HF|walker>

    # two-body double excitations
    e2_2_1 = e2_0 * gt2g
    lt2g = oe.contract("gij,ij->g", chol, t2_green, backend="jax")
    e2_2_2_1 = -oe.contract("g,g->", lt2g, trgl, backend="jax")

    def scan_aux(carry, x):
        chol_i, gl_i = x
        lt2_green_i = oe.contract("pr,qr->pq",chol_i,t2_green,backend="jax")
        carry[0] += 0.5 * oe.contract("pq,pq->",gl_i,lt2_green_i,backend="jax")
        glgp_i = oe.contract("iq,qa->ia", gl_i[:nocc,:],greenp,backend="jax")
        l2t2_1 = oe.contract("ia,jb,iajb->",glgp_i,glgp_i,ci2,backend="jax")
        l2t2_2 = oe.contract("ib,ja,iajb->",glgp_i,glgp_i,ci2,backend="jax")
        carry[1] += 2 * l2t2_1 - l2t2_2
        return carry, 0.0

    [e2_2_2_2, e2_2_3], _ = lax.scan(scan_aux, [0.0, 0.0], (chol, gl))

    e2_2_2 = 4 * (e2_2_2_1 + e2_2_2_2)
    e2_2 = e2_2_1 + e2_2_2 + e2_2_3

    olp = (1.0 + 2*ci1g +  gt2g) # <(1+c1+c2)psi|walker> / <psi|walker> 
    # <exp(T1)HF|h1+h2|walker> + <exp(T1)ci1 HF|(h1+h2)|walker> + <exp(T1)ci2 HF|(h1+h2)|walker>
    e = h0 + (e1_0 + e2_0 + e1_1 + e2_1 + e1_2 + e2_2) / olp

    return olp, e

In [125]:
# walker = jnp.eye(trial.norb)[:,:trial.nelec[0]]
walker = np.random.rand(*walker.shape)
# wave_data['mo_t'] = jnp.eye(trial.norb)[:,:trial.nelec[0]]
# print(walker)
t1 = wave_data['t1'] * 100
t2 = wave_data['t2'] *100 #+ jnp.einsum("ia,jb->iajb", t1, t1)
wave_data['mo_t'] = np.random.rand(*walker.shape)
olp_ad, e_ad = _calc_energy_ad(trial, walker, ham_data, wave_data, t1, t2)
olp_tc, e_tc = _calc_energy_tc(trial, walker, ham_data, wave_data, t1, t2)
print(olp_ad, e_ad)
print(olp_tc, e_tc)

0.3825962460805262 -24.757894186239874
0.382596246080526 -24.757894186241607
